In [136]:
ALPHAVANTAGE_API_KEY = 'Q4L3OK0NNB37YQQZ'

import pandas as pd
import numpy as np
import yfinance as yf 
import glob
import requests
import json
import ast
import datetime as dt

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [47]:
def json_to_csv(json):
    data_list = []
    for url in json:
        data = {}
        for prop in url:
            if type(prop) == str:
                data['url'] = prop
            else:
                data.update(prop)
        data_list.append(data)

    return pd.DataFrame(data_list)

In [132]:
with open('data/etf-processing/etf_scraped.json', 'r') as f:
    url_data = json.load(f)

df = json_to_csv(url_data)

df['isin'] = df['url'].str.split('=').str[1]
df = df.drop(['exchanges', 'factsheet_de', 'kid_en', 'prospectus_de', 'semi-annual report_de', 'annual report_de', 'factsheet_en', 'kid_de', 'prospectus_en', 'semi-annual report_en', 'annual report_en',], axis=1)

In [121]:
def string_eval(string):
    if type(string) == str and '%' in string:
        return float(string.strip('%').strip()) / 100.0
    else:
        return string
    
def turn_to_heading(string):
    if type(string) == str:
        return string.strip().replace(' ','_').lower()
    else:
        return string

In [130]:
def explode_column(row):
    if row:
        if type(row) == dict:
            return row
        elif len(row[0]) == 2:
            evaluated_elems = [[f'{col}_{turn_to_heading(elem[0])}', elem[1]] for elem in row]
            return dict(evaluated_elems)
        else:
            return row
    else:
        return np.nan

In [135]:
explodable_columns = ['sectors', 'countries']#, 'exchanges',]# 'historic_dividend']

for col in explodable_columns:
    df[col] = df[col].apply(explode_column)
    temp_df = df[col].apply(pd.Series)
    # df = df.drop([col], axis=1)
    df = pd.concat([df, temp_df], axis=1)

df['holdings'] = df['holdings'].apply(ast.literal_eval)

# for col in df.columns:
#     # df[col] = df[col].apply(ast.literal_eval)
#     df[col] = df[col].apply(lambda x: np.nan if x == [] else x)

In [138]:
def get_ticker(isin):
    url = 'https://query1.finance.yahoo.com/v1/finance/search'

    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.109 Safari/537.36',
    }

    params = dict(
        q=isin,
        quotesCount=1,
        newsCount=0,
        listsCount=0,
        quotesQueryId='tss_match_phrase_query'
    )

    resp = requests.get(url=url, headers=headers, params=params)
    data = resp.json()
    if 'quotes' in data and len(data['quotes']) > 0:
        return data['quotes'][0]['symbol']
    else:
        return None

In [148]:
end = dt.datetime.now()
start = end - dt.timedelta(days=364)
for isin in df['isin'][:5]:
    print(get_ticker(isin))

UEFD.AS
SPEEU.PA
CM9.PA
UEFY.DE
UEFZ.DE


In [149]:
urna = yf.Ticker('UEFD.AS')

In [150]:
urna.info

{'phone': '+41 1 234 4440',
 'maxAge': 86400,
 'priceHint': 2,
 'previousClose': 123.95,
 'open': 123.54,
 'dayLow': 123.54,
 'dayHigh': 123.54,
 'regularMarketPreviousClose': 123.95,
 'regularMarketOpen': 123.54,
 'regularMarketDayLow': 123.54,
 'regularMarketDayHigh': 123.54,
 'trailingPE': 11.109785,
 'volume': 6,
 'regularMarketVolume': 6,
 'averageVolume': 14,
 'averageVolume10days': 21,
 'averageDailyVolume10Day': 21,
 'totalAssets': 231054176,
 'fiftyTwoWeekLow': 101.81,
 'fiftyTwoWeekHigh': 129.47,
 'fiftyDayAverage': 125.1504,
 'twoHundredDayAverage': 116.8614,
 'currency': 'EUR',
 'fundFamily': 'UBS Fund Management (Luxembourg) S.A.',
 'fundInceptionDate': 1318809600,
 'legalType': 'Exchange Traded Fund',
 'exchange': 'AMS',
 'quoteType': 'ETF',
 'symbol': 'UEFD.AS',
 'underlyingSymbol': 'UEFD.AS',
 'shortName': 'UBS (Lux) Fund Solutions - MSCI',
 'longName': 'UBS(Lux)Fund Solutions – MSCI EMU Small Cap UCITS ETF(EUR)A-dis',
 'firstTradeDateEpochUtc': 1453881600,
 'timeZoneFu

In [29]:
msft = yf.Ticker("CH1135202120.SG")

# get all stock info
msft.info

# get historical market data
hist = msft.history(period="1mo")

# show meta information about the history (requires history() to be called first)
msft.history_metadata

# show actions (dividends, splits, capital gains)
msft.actions
msft.dividends
msft.splits
msft.capital_gains  # only for mutual funds & etfs

# show share count
msft.get_shares_full(start="2022-01-01", end=None)

# show financials:
# - income statement
msft.income_stmt
msft.quarterly_income_stmt
# - balance sheet
msft.balance_sheet
msft.quarterly_balance_sheet
# - cash flow statement
msft.cashflow
msft.quarterly_cashflow
# see `Ticker.get_income_stmt()` for more options

# show holders
msft.major_holders
msft.institutional_holders
msft.mutualfund_holders
msft.insider_transactions
msft.insider_purchases
msft.insider_roster_holders

# show recommendations
msft.recommendations
msft.recommendations_summary
msft.upgrades_downgrades

# Show future and historic earnings dates, returns at most next 4 quarters and last 8 quarters by default.
# Note: If more are needed use msft.get_earnings_dates(limit=XX) with increased limit argument.
msft.earnings_dates

# show ISIN code - *experimental*
# ISIN = International Securities Identification Number
msft.isin

# show options expirations
msft.options

# show news
msft.news

# get option chain for specific expiration
# opt = msft.option_chain('YYYY-MM-DD')
# data available via: opt.calls, opt.puts

$CH1135202120.SG: possibly delisted; No price data found  (period=1mo)
$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)
$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)


$CH1135202120.SG: possibly delisted; No price data found  (period=1mo)
$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)
$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)


$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)
$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)


$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)
$CH1135202120.SG: possibly delisted; No price data found  (1d 1925-07-10 -> 2024-06-15)


404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary//CH1135202120.SG?modules=institutionOwnership%2CfundOwnership%2CmajorDirectHolders%2CmajorHoldersBreakdown%2CinsiderTransactions%2CinsiderHolders%2CnetSharePurchaseActivity&corsDomain=finance.yahoo.com&formatted=false&crumb=cA06rb9jjay
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CH1135202120.SG?modules=recommendationTrend&corsDomain=finance.yahoo.com&formatted=false&symbol=CH1135202120.SG&crumb=D8quvXjSNPZ
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CH1135202120.SG?modules=upgradeDowngradeHistory&corsDomain=finance.yahoo.com&formatted=false&symbol=CH1135202120.SG&crumb=D8quvXjSNPZ
CH1135202120.SG: $CH1135202120.SG: possibly delisted; No earnings dates found


[{'uuid': '6a51e96b-5a1b-391f-9c79-75263ee8e440',
  'title': 'Is Riot Blockchain a No-Brainer Buy After the Bitcoin Halving?',
  'publisher': 'Motley Fool',
  'link': 'https://finance.yahoo.com/m/6a51e96b-5a1b-391f-9c79-75263ee8e440/is-riot-blockchain-a.html',
  'providerPublishTime': 1718455980,
  'type': 'STORY',
  'thumbnail': {'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/uUHUg.TGbLpBmsK_eKGP9g--~B/aD0zODA7dz03MjA7YXBwaWQ9eXRhY2h5b24-/https://media.zenfs.com/en/motleyfool.com/65a5955b5ea454697b75006b3d84bbf6',
     'width': 720,
     'height': 380,
     'tag': 'original'},
    {'url': 'https://s.yimg.com/uu/api/res/1.2/TxbQsU35enZ8Z9co9bxNmw--~B/Zmk9ZmlsbDtoPTE0MDtweW9mZj0wO3c9MTQwO2FwcGlkPXl0YWNoeW9u/https://media.zenfs.com/en/motleyfool.com/65a5955b5ea454697b75006b3d84bbf6',
     'width': 140,
     'height': 140,
     'tag': '140x140'}]},
  'relatedTickers': ['RIOT']},
 {'uuid': '4cef431f-74da-3401-b4c8-52623a630c60',
  'title': 'CARsgen Presents First-in-human Resul